<a href="https://colab.research.google.com/github/author-sanjay/AirSafetyAI/blob/Data-Normalization/AirCraftAccidentDataAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import requests
from bs4 import BeautifulSoup
import time
import csv
from google.colab import drive
from collections import defaultdict
import json
import pandas as pd
import re
from datetime import datetime

# Data Collection

###### Data collection has been done already from airsafety db from year 2000 to 2025 resulting up to 6500+ recorded incidents found to train the model. Please note that this data is only being used for research purposes and model training

# Data Processing

#### Finding and Deleting Duplicates in dat

In [8]:


file_path = "/content/drive/MyDrive/accidents.json"

# Load your data
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Dictionary to track occurrences
seen = defaultdict(list)

for idx, entry in enumerate(data):
    # Composite key: Date + Time + Registration + Location
    key = f"{entry.get('Date','')}_{entry.get('Time','')}_{entry.get('Registration','')}_{entry.get('Location','')}"
    seen[key].append(idx)

# Find duplicates (keys with more than 1 entry)
duplicates = {k: v for k, v in seen.items() if len(v) > 1}

print(f"✅ Total entries: {len(data)}")
print(f"⚠️ Potential duplicates found: {len(duplicates)}")

✅ Total entries: 6791
⚠️ Potential duplicates found: 0


#### Data Normalisation

In [9]:


file_path = "/content/drive/MyDrive/accidents.json"
cleaned_file_path = "/content/drive/MyDrive/accidents_cleaned.json"
csv_file_path = "/content/drive/MyDrive/accidents_cleaned.csv"

# --- Load JSON into pandas ---
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)


###### Fixing Unknown Dates

In [10]:


# --- Handle unk. date YYYY ---
def normalize_date(val):
    if pd.isna(val):
        return None
    val = str(val).strip()

    # If format is "unk. date YYYY"
    match = re.match(r"unk\. date (\d{4})", val, flags=re.IGNORECASE)
    if match:
        year = int(match.group(1))
        return f"{year}-12-31"

    # Try normal parsing
    try:
        return pd.to_datetime(val, errors="coerce").strftime("%Y-%m-%d")
    except Exception:
        return None

df["Date"] = df["Date"].apply(normalize_date)
print(df.head())


         Date      Time                            Type  \
0  2000-01-01  13:00 LT          Cessna 550 Citation II   
1  2000-01-03             Beechcraft 200 Super King Air   
2  2000-01-04  17:25 LT  Beechcraft B200 Super King Air   
3  2000-01-05     13:25  Embraer EMB-110P1A Bandeirante   
4  2000-01-07                             Antonov An-26   

                    Owner/operator Registration       MSN Total airframe hrs  \
0               US Customs Service       N752CC  550-0018        12159 hours   
1  Kalahari Air Services & Charter       A2-AEZ    BB-421                NaN   
2                          Private       N895TT   BB-1239         3238 hours   
3         Skypower Express Airways       5N-AXL    110455                NaN   
4                          Unknown       D2-FBR      7206                NaN   

                     Engine model                     Fatalities  \
0                     P&W JT15D-4   Fatalities: 0 / Occupants: 3   
1                           

Normalizing Time

In [12]:
def normalize_time(val):
    val = str(val).strip()
    if not val:
        # missing or empty → set to noon
        return "12:00"

    # Some have "LT" suffix
    val = val.replace("LT", "").strip()

    try:
        t = pd.to_datetime(val, format="%H:%M", errors="coerce")
        if pd.isna(t):
            return "12:00"   # fallback if weird format
        return t.strftime("%H:%M")
    except:
        return "12:00"

df["Time"] = df["Time"].apply(normalize_time)
print(df.head())

         Date   Time                            Type  \
0  2000-01-01  13:00          Cessna 550 Citation II   
1  2000-01-03  12:00   Beechcraft 200 Super King Air   
2  2000-01-04  17:25  Beechcraft B200 Super King Air   
3  2000-01-05  13:25  Embraer EMB-110P1A Bandeirante   
4  2000-01-07  12:00                   Antonov An-26   

                    Owner/operator Registration       MSN Total airframe hrs  \
0               US Customs Service       N752CC  550-0018        12159 hours   
1  Kalahari Air Services & Charter       A2-AEZ    BB-421                NaN   
2                          Private       N895TT   BB-1239         3238 hours   
3         Skypower Express Airways       5N-AXL    110455                NaN   
4                          Unknown       D2-FBR      7206                NaN   

                     Engine model                     Fatalities  \
0                     P&W JT15D-4   Fatalities: 0 / Occupants: 3   
1                             NaN      Fatalit

Merging Time and Date into one column

In [13]:
df["Datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    errors="coerce"   # in case something slips through
)

# Step 4: drop old columns
df.drop(columns=["Date", "Time"], inplace=True)

Convert to UTC

In [14]:
df["Datetime"] = df["Datetime"].dt.tz_localize("UTC")

Normalizinng Categories

In [16]:
print(df["Category"].unique())

['Accident' 'UK' 'Other' 'Unlawful Interference' nan 'Incident'
 'Serious incident']


In [17]:
import numpy as np

def normalize_category(cat):
    if pd.isna(cat):  # handle NaN
        return "Incident"   # fallback instead of dropping
    cat = str(cat).strip().lower()
    if cat == "accident":
        return "Accident"
    elif cat == "incident":
        return "Incident"
    elif cat == "serious incident":
        return "Serious Incident"
    elif cat == "unlawful interference":
        return "Unlawful Interference"
    elif cat == "uk":
        return "Unknown"
    elif cat == "other":
        return "Other"
    else:
        return "Incident"  # fallback if anything weird pops up

df["Category"] = df["Category"].apply(normalize_category)
print(df["Category"].unique())


['Accident' 'Unknown' 'Other' 'Unlawful Interference' 'Incident'
 'Serious Incident']


Normalizing Operator

In [19]:
print(df["Owner/operator"].unique())

['US Customs Service' 'Kalahari Air Services & Charter' 'Private' ...
 'Swan River Seaplanes' 'Foxtrot Jet LLC'
 'Corporate Flight Management Inc dba Contour Aviation']


In [24]:
def normalize_operator(op):
    if pd.isna(op):
        return "Private"
    op = str(op).strip()
    if op.lower() == "unknown":
        return "Private"
    if op.lower() == "":
        return "Private"
    if op.lower() == "private":
        return "Private"
    return op  # keep airline name as is

df["Owner/operator"] = df["Owner/operator"].apply(normalize_operator)
print(df["Owner/operator"].nunique())
print(df["Owner/operator"].value_counts().head(20))


3532
Owner/operator
Private                                                  315
Delta Air Lines                                          115
American Airlines                                        110
United Airlines                                          104
Southwest Airlines                                        84
United States Air Force - USAF                            45
Ryanair                                                   41
Northwest Airlines                                        35
Russian Air Force                                         28
Lufthansa                                                 25
British Airways                                           24
American Eagle Airlines                                   24
Air France                                                23
Air Canada                                                23
Al Quwwat al-Jawwiya As-Sudaniya (Sudanese Air Force)     21
Pakistan International Airlines - PIA                     20
FedE

Normalizing MSN and Registration

In [25]:
def normalize_empty(val):
    if pd.isna(val) or str(val).strip() == "":
        return "Unknown"
    return str(val).strip()

df["Registration"] = df["Registration"].apply(normalize_empty)
df["MSN"] = df["MSN"].apply(normalize_empty)

print(df[["Registration", "MSN"]].head(10))


  Registration        MSN
0       N752CC   550-0018
1       A2-AEZ     BB-421
2       N895TT    BB-1239
3       5N-AXL     110455
4       D2-FBR       7206
5       C-GXVX       B-18
6       HL7441  20373/168
7       HB-AKK   340B-213
8       N909AW  24522/252
9       HB-AAM    SH.3763
